### Set up environment

In [1]:
%pip install accelerate bitsandbytes google-cloud-aiplatform peft trl scikit-learn tokenizers torch transformers unsloth evaluate python-dotenv sentencepiece protobuf matplotlib

Note: you may need to restart the kernel to use updated packages.


### import library

In [2]:
import os
import sys
import torch
import evaluate
import numpy as np
from trl import SFTTrainer
from tqdm.auto import tqdm
from torch.optim import AdamW
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
from accelerate import Accelerator
from torch.utils.data import DataLoader
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments

/home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to load /home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/halogenbr/.venvs/tic-unsloth312/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


### Login huggingface

In [3]:
load_dotenv()
hf_token = os.getenv("HUGFACE_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face successfully.")
else:
    print("HUGFACE_TOKEN not found in environment variables. Please set it to log in.")

Logged in to Hugging Face successfully.


### Device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Load data and preproces

In [5]:
print("Loading dataset...")
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
print("Dataset loaded successfully.")
print("Dataset structure:")
print(dataset)

Loading dataset...
Dataset loaded successfully.
Dataset structure:
Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 15011
})


In [6]:
def format_prompt(example):
    instruction = example["instruction"]
    context = example["context"]
    response = example["response"]
    text = []
    for instruction, context, response in zip(instruction, context, response):
        prompt = f"<|im_start|>user{instruction}\n{context}<|im_end|>\n<|im_start|>assistan\n{response}<|im_end|>"
        text.append(prompt)
    return {"text": text}
dataset = dataset.map(format_prompt, batched=True)

### load model and training

In [7]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
)


dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_data = dataset_split["train"]
eval_data = dataset_split["test"]

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    processing_class=tokenizer,
    args=TrainingArguments(
        output_dir=model_name.split("/")[-1] + "-lora-finetuned",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        save_strategy="steps",
        save_steps=10,
        eval_strategy="steps",
        eval_steps=10,
        logging_strategy="steps",
        load_best_model_at_end=True, 
        save_total_limit=1,
        optim="adamw_torch",           
        lr_scheduler_type="cosine",
        learning_rate=2e-5,
        weight_decay=0.01,
        push_to_hub=True,
        max_steps=20,
        # save_steps=10,
    )
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,No log,2.198159,2.186399,14072.000000,0.539557
20,No log,2.187824,2.207206,28518.000000,0.540404


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=20, training_loss=2.3512805938720702, metrics={'train_runtime': 2277.6461, 'train_samples_per_second': 0.07, 'train_steps_per_second': 0.009, 'total_flos': 93037761603072.0, 'train_loss': 2.3512805938720702, 'epoch': 0.011843079200592153})

### Save model

In [8]:
trainer.push_to_hub("qwen-2.5b-finetuned")

Upload 0 LFS files: 0it [00:00, ?it/s]
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/HalogenFlo/Qwen2.5-0.5B-Instruct-lora-finetuned/commit/83b6818e94038827c83619d1af263fdde6c7ab53', commit_message='qwen-2.5b-finetuned', commit_description='', oid='83b6818e94038827c83619d1af263fdde6c7ab53', pr_url=None, repo_url=RepoUrl('https://huggingface.co/HalogenFlo/Qwen2.5-0.5B-Instruct-lora-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='HalogenFlo/Qwen2.5-0.5B-Instruct-lora-finetuned'), pr_revision=None, pr_num=None)

### Test model

In [9]:

def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = trainer.model.generate(**inputs, max_length=2048, do_sample=True, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response
test_prompt = "What is the capital of France?"
response = generate_response(test_prompt)
print("Prompt:", test_prompt)
print("Response:", response)

Prompt: What is the capital of France?
Response: What is the capital of France? The capital of France is Paris. The city has a population of 12 million people and covers an area of 784 km². It is located on the River Seine, which runs through it for much of its length.

Paris was founded in 635 AD by the Romans. It became the capital of France when Charles Martel united the Frankish Kingdoms under his rule in 790 AD. Paris is also home to many important museums, including the Louvre Museum, where famous paintings are displayed. Paris is also home to many other historic landmarks such as Notre Dame Cathedral, Eiffel Tower, and the Pont des Arts. 

In addition to being the capital of France, Paris is home to many universities, including the University of Paris, where the French Academy of Sciences was founded in 1810. Paris is home to several international organizations, including the European Union, the United Nations, and the World Trade Organization. 

The city is also home to many cu